![img](https://github.com/JuliaLang/julia/raw/master/doc/src/assets/logo.svg)![img](https://avatars.githubusercontent.com/u/7346142?s=200&v=4)

# 1. Installation and Setup

JuliaGPU packages are easy to install: Just do `Pkg.add("Package Name")` to install the package.

In [ ]:
using Pkg
Pkg.add(["BenchmarkTools", "KernelAbstractions", "CUDA", "AMDGPU", "Metal"])

In [ ]:
using BenchmarkTools, LinearAlgebra, KernelAbstractions

In [ ]:
Pkg.status()

## 1.1. Selecting a backend

GPU backend packages provide bindings to NVIDIA's CUDA or respective other vendor. CUDA.jl provides all of the compiler and runtime logic needed to program NVIDIA GPUs; the only thing you need to provide is a functional NVIDIA driver (which most HPC systems already have installed and configured), but you don't need to install the CUDA toolkit! CUDA.jl downloads one if it's not already available on your system (again, HPC systems usually have this provided for you):

In [ ]:
# add all options for different backends and array types
DEV = :AMDGPU

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal 
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
else DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
end
#allocate(backend, Float32, res, res)

# 2. Example: Vector Addition

As a simple example, let's take a look at vector addition. Let's assume you have two vectors $\vec{a}$ and $\vec{b}$ and you want to add them elementwise. You can do this in many ways in Julia: 
1. simple for loop on a CPU
2. julia array add (+) on a CPU or GPU
3. GPU kernel programming in KernelAbstractions

In [ ]:
# define our input a, b vectors, and output c vector in CPU RAM
vector_size = 1024
a = rand(1:4, vector_size)
b = rand(1:4, vector_size)
c = zeros(Int, vector_size)

# what's in a?
a

## 2.1. CPU vector addition

Let's write a simple CPU loop to add two vectors in serial:

In [ ]:
# Note: the exclamation mark (!) doesn't do anything special
# It's just used to indicate that a function mutates its arguments
function vadd!(a, b, c)
    for i in 1:length(c)
        c[i] = a[i] + b[i]
    end
end
vadd!(a, b, c)
c

Thankfully, Julia has a ton of built-in array operations, so we don't actually need to implement this ourselves. Julia's vector add (+) operation works exactly as you'd expect:

In [ ]:
c = a + b

# Note that, unlike `vadd!`, the above operation allocates a new `c` as the output vector - this is important to remember.

## 2.2. GPU arrays and array operations

Let's do the same on the GPU

In [ ]:
# We need first to make copies of the a and b vectors on the GPU, and define a new dc empty GPU vector

# allocate reserves new GPU memory of that backend (uninitialized). copyto! then copies the data from the CPU

da = allocate(Backend, eltype(a), size(a)...)
copyto!(da, a)
db = allocate(Backend, eltype(b), size(b)...)
copyto!(db, b)

# KernelAbstractions.zeros allocates a new device array, and initializes it with Int64 zeros
dc = KernelAbstractions.zeros(Backend, Int, size(a)...)

# We can also safely take a look at what's in da, even though it's on the GPU!
# It's a different array type, and that fact is made clear to us:
da

Now that we know how to allocate on the GPU, let's see how to use this same add (+) operation on the GPU to add two of those vectors using CUDA:

In [ ]:
dc = da + db

# We can add GPU vectors using the same `+` operator, thanks to Julia's multiple dispatch!
# Also, like for the CPU add operation, this one also allocates a new GPU array for output `dc`

## 2.3. Writing a KernelAbstractions kernel

Let us learn how to write our own GPU kernels in pure Julia.

In array operations, GPUs can leverage implicit parallelism (expressed over the array's elements) to automatically execute these operations in parallel on a GPU. When using hand-rolled kernels, it is instead the programmer's responsibility to decide how to effectively assign the available parallel execution resources for the specific operation. Let's see how this is done for vector addition.

In [ ]:
# First we define the kernel. @kernel marks this function so KernelAbstractions
# can compile it for the selected backend.

@kernel function vadd_kernel!(c, a, b)
    # @index(Local) is this thread's index within its workgroup (one workgroup == one block).
    # With a single workgroup, that index maps directly onto the elements of a and b.
    i = @index(Local)
    # Each thread adds its own element into c
    c[i] = a[i] + b[i]

    # Kernels don't return anything
    return
end

# Calling the function is not enough to run it. KernelAbstractions splits this into two steps:
# 1. compile the kernel for our backend and a workgroup size (how many threads run together)
# 2. launch it, passing ndrange: the total number of threads to run
# Let's start with a single workgroup that covers the whole vector.

kernel! = vadd_kernel!(Backend, length(da))
kernel!(dc, da, db, ndrange=length(da))

# Careful: launching the kernel only tells the device to start it. There is no guarantee it has finished!
# Wait until the kernel completes before we read the result.

KernelAbstractions.synchronize(Backend)

dc

When we launched `vadd_kernel!`, every element of the vector got its own thread, and all of those threads sat in one workgroup. The pictures below show what that launch looks like on the chip.

## 2.4. CPU vs GPU

![CPU versus GPU: a few large cores with control and cache on the left, a wide grid of small arithmetic units on the right](img/CPUvsGPU.png)

A CPU is built around a few powerful cores. Most of the chip is control logic and cache, with a handful of arithmetic units (the green ALUs) and DRAM underneath. It runs a small number of complex instruction streams well. That is how the serial `vadd!` loop works: one core steps through the vector and computes one sum at a time.

A GPU spends that silicon on arithmetic instead. The right-hand side of the picture is mostly a grid of small ALUs, with less control and cache attached to each one, and device DRAM along the bottom. `vadd_kernel!` maps onto that layout. Each thread runs the same body, `c[i] = a[i] + b[i]`, and the launch spreads those threads across many ALUs at once.

## 2.5. Threads, blocks, and streaming multiprocessors

![GPU execution hierarchy: threads inside a thread block with shared memory, blocks gathered into clusters, and a grid over global memory](img/threadsGPU.png)

The `vadd_kernel!` launch is a **grid** of **threads**, grouped into **thread blocks**. KernelAbstractions calls a thread block a workgroup. The picture shows the memory each of those groups can see. The hardware that runs a block of `vadd_kernel!` is a streaming multiprocessor.

- A **thread** runs `vadd_kernel!` once. It reads one index from `@index(Local)` and performs `c[i] = a[i] + b[i]`. That index, and the values loaded for it, stay in the thread's registers and a small private local memory.
- A **thread block** is the group of `vadd_kernel!` threads scheduled together. Threads in the same block share a region of **shared memory** (the yellow box). Shared memory sits next to the arithmetic units, so reusing a value there is much faster than reading it back from DRAM, and threads in the block can wait for each other. This kernel never uses that shared region: each thread only touches its own `a[i]` and `b[i]`. The launch still chooses a block size, and we compiled `vadd_kernel!` as one block of `length(da)` threads.
- A **streaming multiprocessor (SM)** is the unit on the chip that runs that block of `vadd_kernel!`. A GPU contains many SMs, and the scheduler places each block onto one SM for the whole time that block runs. An SM can hold several blocks at once when it still has free registers and shared memory. It also caps how many threads a single block may contain, which is the limit the next example runs into: `vadd_kernel!` on a vector of 10240 elements, still packed into that one block.
- All of the blocks in one `vadd_kernel!` launch form the **grid**, and every thread in the grid can read and write **global memory**. That is where `da`, `db`, and `dc` live, so each thread of the kernel can load its inputs and store its sum. Registers and shared memory stay inside a block, so anything that must be visible across blocks is written to global memory. With one block, this launch's grid is just that block.

The middle of the picture is a **thread block cluster**: several blocks that can reach one another's shared memory, called distributed shared memory. `vadd_kernel!` does not use clusters. It uses one thread per element, one block, and the grid.

## 2.6. Multiple blocks and global indexing

The first `vadd_kernel!` put every element in one thread block, and that block runs on one streaming multiprocessor. Growing the vector grows that block by the same amount. Let's scale the vectors to 10240 elements and launch `vadd_kernel!` the same way: one workgroup, one thread per element.

In [ ]:
vector_size = 10240
a = rand(1:4, vector_size)
b = rand(1:4, vector_size)

# Allocate device memory for the longer vectors and copy the CPU data over,
# same as for the first vadd_kernel! launch.
da = allocate(Backend, eltype(a), size(a)...)
copyto!(da, a)
db = allocate(Backend, eltype(b), size(b)...)
copyto!(db, b)
dc = KernelAbstractions.zeros(Backend, Int, vector_size)

# One workgroup again, so the block of vadd_kernel! is now 10240 threads on one SM.
kernel! = vadd_kernel!(Backend, length(da))
kernel!(dc, da, db, ndrange=length(da))

That launch of `vadd_kernel!` failed for the reason in the picture above. One block sits on one streaming multiprocessor, and an SM only accepts a limited number of threads in that block. A block of 10240 threads is past the cap:

In [ ]:
# The workgroup size passed to vadd_kernel!(Backend, workgroupsize) is the number of
# threads in one block. On NVIDIA GPUs an SM caps that block at 1024 threads,
# which is why the 10240-thread workgroup was rejected.
max_threads_per_block = 1024

Since 10240 > 1024, one SM cannot run this `vadd_kernel!` launch as a single block. A GPU has many SMs, and each SM can take its own block, so the 10240 threads can be split into blocks of 1024. That is `cld(10240, 1024)` blocks, ten blocks the GPU can place on its SMs, instead of one block no SM will accept.

`@index(Local)`, which the first `vadd_kernel!` uses, is only the index inside the current block. Once the launch spans more than one block, that index starts over in every block, and threads on different SMs would write the same `c[i]`. The index has to count across blocks.

Across blocks, the element index for `vadd_kernel!` is

`i = @index(Local) + (@index(Group) - 1) * workgroupsize`

- `@index(Local)` is this thread's index inside its block, starting at 1. In the one-block launch of `vadd_kernel!`, that was enough to name every element.
- `@index(Group)` is this block's index in the grid, also starting at 1. Each of those blocks is the unit placed on an SM.
- `workgroupsize` is how many threads are in the block. We will set it to 1024, the per-block cap, so a vector of 10240 elements fills ten blocks.

`(@index(Group) - 1) * workgroupsize` is the first element of the 10240-element vector that this block owns. Adding `@index(Local)` is that thread's element.

KernelAbstractions computes the same index as `@index(Global)`. With one block, `@index(Global)` and `@index(Local)` are the same number, which is why the first `vadd_kernel!` was correct. With several blocks they differ, so the kernel below must use `@index(Global)`.

Rewrite `vadd_kernel!` so each thread picks its element with `@index(Global)`. The last block of the 10240-element vector can contain a few threads past the end of `c`, so those threads must skip the write.

In [ ]:
@kernel function vadd_kernel!(c, a, b)
    # Global index of this thread across every block of the launch.
    # Same value as @index(Local) + (@index(Group) - 1) * workgroupsize.
    i = @index(Global)

    # The last block can have a few threads past the end of the vector.
    if i <= length(c)
        c[i] = a[i] + b[i]
    end

    return
end

Launch this `vadd_kernel!` with a workgroup of 1024 threads, the most one SM will take in a block. `ndrange=length(da)` covers all 10240 elements, so KernelAbstractions launches `cld(10240, 1024)` blocks and the device places each block on an SM.

In [ ]:
# 1024 threads per block of vadd_kernel!. The number of blocks is
# cld(length(da), workgroupsize): division rounded up.
workgroupsize = max_threads_per_block
kernel! = vadd_kernel!(Backend, workgroupsize)
kernel!(dc, da, db, ndrange=length(da))

KernelAbstractions.synchronize(Backend)
dc

# 3. Tutorial completed

Congratulations — you finished the KernelAbstractions GPU introduction.

Along the way you:

- set up a portable `Backend` and moved data with `allocate`, `copyto!`, and `KernelAbstractions.zeros`
- wrote `vadd_kernel!`, then grew it from one workgroup to many blocks with `@index(Global)`
- connected that launch to the hardware picture: threads, blocks, and streaming multiprocessors

Next, continue with **`matrix_multiplication.ipynb`** for CPU and GPU matrix multiplication, benchmarking across sizes, the memory hierarchy, and a shared-memory tiled kernel.

